Importando librerias necesarias

In [1]:
import yaml
import pandas as pd
import numpy as np

# Plotting Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA

import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
import mlflow.pyfunc

# Setting Parent Folder
%cd ..

# Local Functions
from src.data.data_transformation import data_transformer

print('Libraries loaded')

c:\Users\meo1slp\Desktop\ITESM MAI\MLOps\itesm_tc5044_10_mlops_equipo22
Libraries loaded


c:\Users\meo1slp\.conda\envs\lfs_knn-dtw_missingcrimpingfinger\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## 1 Config

In [2]:
with open('params.yaml') as conf_file:
    config = yaml.safe_load(conf_file)

print(config)

{'base': None, 'data': {'input_data': 'data/Steel_industry_data.csv'}, 'train': {'test_size': 0.2, 'random_state': 42, 'activation': 'relu', 'activation_2': 'softmax', 'optimizer': 'adam', 'loss': 'sparse_categorical_crossentropy', 'epochs': 50, 'batch_size': 10, 'verbose': 1, 'axis': -1}, 'reports': {'model': 'models/steel_industry_model.keras'}}


## 2 Load dataset

In [42]:
data = pd.read_csv(config['data']['input_data'])
data.head(20)

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load
5,01/01/2018 01:30,3.28,3.56,0.0,0.0,67.76,100.0,5400,Weekday,Monday,Light_Load
6,01/01/2018 01:45,3.60,4.14,0.0,0.0,65.62,100.0,6300,Weekday,Monday,Light_Load
7,01/01/2018 02:00,3.60,4.28,0.0,0.0,64.37,100.0,7200,Weekday,Monday,Light_Load
8,01/01/2018 02:15,3.28,3.64,0.0,0.0,66.94,100.0,8100,Weekday,Monday,Light_Load
9,01/01/2018 02:30,3.78,4.72,0.0,0.0,62.51,100.0,9000,Weekday,Monday,Light_Load


In [23]:
data.dtypes

date                                     object
Usage_kWh                               float64
Lagging_Current_Reactive.Power_kVarh    float64
Leading_Current_Reactive_Power_kVarh    float64
CO2(tCO2)                               float64
Lagging_Current_Power_Factor            float64
Leading_Current_Power_Factor            float64
NSM                                       int64
WeekStatus                               object
Day_of_week                              object
Load_Type                                object
dtype: object

In [45]:
data = data.set_index('date')

Transforamción de Datos

In [4]:
#X, y = data_transformer(data)

## 3 Training Model

In [ ]:
type(config['train']['test_size'])
type(config['train']['random_state'])

In [5]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = config['train']['test_size'], random_state = config['train']['random_state'])

In [13]:
model = RandomForestClassifier(max_depth=None,min_samples_split=2, n_estimators=200)
model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200)

In [14]:
# Make predictions and evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.93050799086758


### 3.1 Local Prediction

In [46]:
y = data[['Load_Type']]
X = data.drop(['Load_Type'], axis=1)

In [47]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = config['train']['test_size'], random_state = config['train']['random_state'])

In [49]:
X_train.head()

,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week
date,,,,,,,,,
07/04/2018 17:15,4.18,0.00,21.78,0.00,100.00,18.85,62100,Weekend,Saturday
30/05/2018 02:45,2.92,4.93,0.00,0.00,50.96,100.00,9900,Weekday,Wednesday
23/02/2018 01:45,6.44,7.34,0.00,0.00,65.95,100.00,6300,Weekday,Friday
13/12/2018 13:30,56.92,22.28,0.00,0.03,93.12,100.00,48600,Weekday,Thursday
15/04/2018 02:45,3.10,3.82,0.00,0.00,63.01,100.00,9900,Weekend,Sunday


In [50]:
y_train.head()

,Load_Type
date,
07/04/2018 17:15,Medium_Load
30/05/2018 02:45,Light_Load
23/02/2018 01:45,Light_Load
13/12/2018 13:30,Medium_Load
15/04/2018 02:45,Light_Load


In [17]:
from joblib import dump

In [52]:
def param_types(input_dataframe):
    #input_dataframe['date'] = pd.to_datetime(input_dataframe['date'], format="%d/%m/%Y %H:%M")
    numeric_variables = input_dataframe.select_dtypes(include='number').columns.to_list()
    object_variables = input_dataframe.select_dtypes(include = 'object').columns.to_list()
    return numeric_variables, object_variables

In [53]:
param_types(X)[1]

['WeekStatus', 'Day_of_week']

In [54]:
X_train_param_types = param_types(X_train)
X_test_param_types = param_types(X_test)

In [55]:
X_train[X_train_param_types[0]]

,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM
date,,,,,,,
07/04/2018 17:15,4.18,0.00,21.78,0.00,100.00,18.85,62100
30/05/2018 02:45,2.92,4.93,0.00,0.00,50.96,100.00,9900
23/02/2018 01:45,6.44,7.34,0.00,0.00,65.95,100.00,6300
13/12/2018 13:30,56.92,22.28,0.00,0.03,93.12,100.00,48600
15/04/2018 02:45,3.10,3.82,0.00,0.00,63.01,100.00,9900
...,...,...,...,...,...,...,...
25/06/2018 12:45,5.76,0.00,25.27,0.00,100.00,22.22,45900
07/03/2018 06:30,3.71,4.57,0.00,0.00,63.03,100.00,23400
28/04/2018 13:15,13.57,0.14,14.33,0.01,99.99,68.76,47700


In [56]:
# Scaling
scaler = StandardScaler()

X_train_scale = scaler.fit_transform(X_train[X_train_param_types[0]]) # Numeric Variables
X_test_scale = scaler.transform(X_test[X_test_param_types[0]]) # Numeric Variables

#dump(scaler, config['model_ridge']['scaler_path'])

In [57]:
#Aplicando reducción de dimensionalidad
pca = PCA(n_components=0.95)  
X_train_pca = pca.fit_transform(X_train_scale)
X_test_pca = pca.transform(X_test_scale)

#dump(pca, config['model_ridge']['scaler_path'])

In [58]:
X_train_pca

array([[-2.67000942,  2.46999845, -0.48827099,  0.41420423],
       [-0.67287892, -2.30189239, -0.33272737,  0.29539217],
       [-0.52005036, -1.88494913, -0.6120434 , -0.22853485],
       ...,
       [-1.28008376,  1.28754788, -0.3057479 , -0.51811777],
       [-0.57757165, -0.24948005,  2.0434161 ,  0.06757036],
       [-2.40764959,  1.92456955, -0.83030795,  0.08458058]])

In [59]:
X_train_pca_df = pd.DataFrame(X_train_pca, columns=[f'PC{i+1}' for i in range(X_train_pca.shape[1])])
X_train_pca_df

,PC1,PC2,PC3,PC4
0,-2.670009,2.469998,-0.488271,0.414204
1,-0.672879,-2.301892,-0.332727,0.295392
2,-0.520050,-1.884949,-0.612043,-0.228535
3,1.698346,0.557963,0.067293,-0.476807
4,-0.680649,-1.950280,-0.408159,-0.195679
...,...,...,...,...
28027,-2.781668,2.330386,-1.155876,0.379753
28028,-0.640403,-1.687301,0.044127,-0.057241
28029,-1.280084,1.287548,-0.305748,-0.518118
28030,-0.577572,-0.249480,2.043416,0.067570


In [60]:
#Codificación de variables categóricas
X_train_encoded_df = pd.get_dummies(X_train[X_train_param_types[1]], columns=X_train_param_types[1], drop_first=True) # Objective Variables
X_train_encoded_df

,WeekStatus_Weekend,Day_of_week_Monday,Day_of_week_Saturday,Day_of_week_Sunday,Day_of_week_Thursday,Day_of_week_Tuesday,Day_of_week_Wednesday
date,,,,,,,
07/04/2018 17:15,1,0,1,0,0,0,0
30/05/2018 02:45,0,0,0,0,0,0,1
23/02/2018 01:45,0,0,0,0,0,0,0
13/12/2018 13:30,0,0,0,0,1,0,0
15/04/2018 02:45,1,0,0,1,0,0,0
...,...,...,...,...,...,...,...
25/06/2018 12:45,0,1,0,0,0,0,0
07/03/2018 06:30,0,0,0,0,0,0,1
28/04/2018 13:15,1,0,1,0,0,0,0


In [41]:

#Concatenando nuevo dataframe
X_train_transformed = pd.concat([X_train_encoded_df, X_train_pca_df], axis=1)
X_train_transformed

,WeekStatus_Weekend,Day_of_week_Monday,Day_of_week_Saturday,Day_of_week_Sunday,Day_of_week_Thursday,Day_of_week_Tuesday,Day_of_week_Wednesday,PC1,PC2,PC3,PC4
9284,1.0,0.0,1.0,0.0,0.0,0.0,0.0,-2.536410,2.688208,0.050998,0.477069
14314,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.627921,-2.404616,-0.255366,0.550050
5094,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.006434,0.774279,0.469968,0.085757
33269,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN
9994,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.032911,0.352096,0.208578,-0.680334
...,...,...,...,...,...,...,...,...,...,...,...
28005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.271889,-1.215999,-1.001302,-0.544749
28008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.614776,-1.545411,-0.204135,-0.406341
28009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.514850,1.213006,0.753325,-0.763066
28015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.667599,-2.304680,0.224140,0.790054


In [ ]:
# Training
classifier = RidgeClassifier(alpha=1000)
classifier.fit(X_train_transform_scale, ytrain.iloc[:, 0])
#dump(classifier, artifacts['classifier_path'])

In [ ]:
model = RandomForestClassifier(max_depth=None,min_samples_split=2, n_estimators=200)
model.fit(X_train, y_train)
#dump(model, artifacts['classifier_path'])

In [ ]:
data['date'] = pd.to_datetime(data['date'], format="%d/%m/%Y %H:%M")
data['month'] = data['date'].dt.month_name()
numeric_variables = data.select_dtypes(include='number').columns.to_list()
object_variables = data.select_dtypes(include = 'object').columns.to_list()
object_variables.remove('Load_Type')

#'Scaling' la data numérica
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data[numeric_variables])

#Aplicando reducción de dimensionalidad
pca = PCA(n_components=0.95)  
data_pca = pca.fit_transform(data_scaled)

cantidad_variables = data_pca.shape[1]
#print(f'Se redujo a {cantidad_variables} variables numéricas')
pca_df = pd.DataFrame(data_pca, columns=[f'PC{i+1}' for i in range(data_pca.shape[1])])

#Codificación de variables categóricas
data_encoded_df = pd.get_dummies(data[object_variables], columns=object_variables, drop_first=True)
#Concatenando nuevo dataframe
X = pd.concat([data_encoded_df, pca_df], axis=1)
#Variable objetivo codificada
le = LabelEncoder()
y = le.fit_transform(data['Load_Type'])

### 3.2 PyFunc

In [ ]:
class ModelWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self):
        self.model = None
        self.min_max_scaler = None
        self.minirocket = None

    

### 4. Log Model to MLflow

In [17]:
# Set up MLFlow experiment
mlflow.set_experiment("Wine_Quality_Experiment")

with mlflow.start_run():
    # Train a Logistic Regression model
    model = RandomForestClassifier(max_depth=None,min_samples_split=2, n_estimators=200)
    model.fit(X_train, y_train)
    
    # Make predictions and evaluate the model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Log parameters and metrics to MLFlow
    mlflow.log_param("model_type", "Random Forest Classifier")
    mlflow.log_param("max_depth", None)
    mlflow.log_param("min_samples_split", 2)
    mlflow.log_param("n_estimators", 200)
    mlflow.log_metric("accuracy", accuracy)
    
    # Log the model
    mlflow.sklearn.log_model(model, "model")
    
    print(f"Model accuracy: {accuracy}")

2024/11/01 22:39:42 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Model accuracy: 0.9316495433789954


## 4 Save Model

In [64]:
model.save(config['reports']['model'])

## 5 Model Prediction

In [ ]:
loaded_model = load_model(config['reports']['model'])

In [ ]:
y_pred_cargado = np.argmax(loaded_model.predict(X_test), axis=-1)
accuracy_cargado = accuracy_score(y_test, y_pred_cargado)
print(f'Accuracy del modelo cargado: {accuracy_cargado:.4f}')